In [19]:
import modal

app = modal.App("leaf-disease-cnn-random-search")

image = (
    modal.Image.debian_slim()
    .pip_install(
        "torch",
        "torchvision",
        "numpy",
        "pandas",
        "scikit-learn",
        "Pillow",
        "tqdm",
    )
    # Make sure this file exists locally next to this notebook.
    .add_local_file("dataset_and_CNN_utils.py", remote_path="/root/dataset_and_CNN_utils.py")
)

# Data volume contains /data/data/train.csv and /data/data/train_images
data_vol = modal.Volume.from_name("plant-pictures", create_if_missing=True)

# Checkpoint/results volume
ckpt_vol = modal.Volume.from_name("plant-models", create_if_missing=True)

DATA_ROOT = "/data/data"
CKPT_ROOT = "/ckpt"


## Prepare train/validation split

This function reads `/data/data/train.csv`, creates a train/validation split, and saves:

- `/data/data/train_split.csv`
- `/data/data/val_split.csv`
- `/data/data/classes.json`
- `/data/data/dataset_metadata.json`


In [20]:
@app.function(image=image, volumes={"/data": data_vol, "/ckpt": ckpt_vol})
def prepare_data():
    import json
    from pathlib import Path

    import pandas as pd
    from sklearn.model_selection import train_test_split
    from sklearn.preprocessing import MultiLabelBinarizer

    try:
        data_vol.reload()
    except Exception:
        pass

    data_root = Path("/data/data")
    train_csv = data_root / "train.csv"
    image_dir = data_root / "train_images"

    df = pd.read_csv(train_csv)

    df["label_list"] = df["labels"].apply(lambda x: str(x).split())

    mlb = MultiLabelBinarizer()
    mlb.fit(df["label_list"])

    train_df, val_df = train_test_split(
        df,
        test_size=0.15,
        random_state=42,
        shuffle=True,
    )

    train_df = train_df.reset_index(drop=True)
    val_df = val_df.reset_index(drop=True)

    train_df.to_csv(data_root / "train_split.csv", index=False)
    val_df.to_csv(data_root / "val_split.csv", index=False)

    classes = list(mlb.classes_)

    with open(data_root / "classes.json", "w") as f:
        json.dump(classes, f, indent=2)

    meta = {
        "num_train": int(len(train_df)),
        "num_val": int(len(val_df)),
        "classes": classes,
        "num_classes": int(len(classes)),
    }

    with open(data_root / "dataset_metadata.json", "w") as f:
        json.dump(meta, f, indent=2)

    try:
        data_vol.commit()
    except Exception:
        pass

    return meta


## Random search function

The function below:

1. Samples random hyperparameter combinations.
2. Trains one CNN per sampled combination.
3. Evaluates validation loss, micro F1, and macro F1.
4. Saves one row per trial to `/ckpt/random_search_results.csv`.
5. Saves the best model checkpoint to `/ckpt/best_random_search_cnn.pt`.

For multi-label classification, `macro_f1` is often more informative than `micro_f1` because rare labels receive equal weight.


In [ ]:
@app.function(
    image=image,
    volumes={"/data": data_vol, "/ckpt": ckpt_vol},
    timeout=60 * 60 * 8,
    gpu="T4",
)
def random_search_cnn(
    n_trials=40,
    epochs_per_trial=1,
    batch_size=64,
    lr=1e-4,
    image_size=224,
    num_workers=8,
    prefetch_factor=2,
    seed=42,
    use_mixed_precision=True,
):
    import itertools
    import json
    import random
    import time
    from pathlib import Path

    import pandas as pd
    import torch
    import torch.nn as nn
    from torch.utils.data import DataLoader
    from torchvision import transforms

    from dataset_and_CNN_utils import MultiLabelImageDataset, make_data_loader, FlexibleCNN

    try:
        data_vol.reload()
        ckpt_vol.reload()
    except Exception:
        pass

    torch.manual_seed(seed)
    random.seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.benchmark = True

    data_root = Path("/data/data")
    ckpt_root = Path("/ckpt")
    ckpt_root.mkdir(parents=True, exist_ok=True)

    image_dir = data_root / "train_images"
    train_csv = data_root / "train_split.csv"
    val_csv = data_root / "val_split.csv"
    classes_path = data_root / "classes.json"

    with open(classes_path, "r") as f:
        classes = json.load(f)
    num_classes = len(classes)

    print(f"Classes: {classes}", flush=True)
    print(f"Number of classes: {num_classes}", flush=True)

    train_loader = make_data_loader(
        train_csv,
        image_dir=image_dir,
        classes=classes,
        batch_size=batch_size,
        image_size=image_size,
        num_workers=num_workers,
        shuffle=True,
        prefetch_factor=prefetch_factor,
    )
    val_loader = make_data_loader(
        val_csv,
        image_dir=image_dir,
        classes=classes,
        batch_size=batch_size,
        image_size=image_size,
        num_workers=num_workers,
        shuffle=False,
        prefetch_factor=prefetch_factor,
    )
    print(f"Train batches: {len(train_loader)}", flush=True)
    print(f"Validation batches: {len(val_loader)}", flush=True)

    # ------------------------------------------------------------------
    # Metrics
    # ------------------------------------------------------------------

    def compute_multilabel_f1_from_logits(logits, targets, threshold=0.5, eps=1e-8):
        preds = (logits >= threshold).float()
        targets = targets.float()

        # Micro F1: aggregate TP, FP, FN globally.
        tp_micro = (preds * targets).sum()
        fp_micro = (preds * (1.0 - targets)).sum()
        fn_micro = ((1.0 - preds) * targets).sum()

        micro_f1 = (2.0 * tp_micro) / (
            2.0 * tp_micro + fp_micro + fn_micro + eps
        )

        # Macro F1: compute per-class F1, then average.
        tp_class = (preds * targets).sum(dim=0)
        fp_class = (preds * (1.0 - targets)).sum(dim=0)
        fn_class = ((1.0 - preds) * targets).sum(dim=0)

        f1_per_class = (2.0 * tp_class) / (
            2.0 * tp_class + fp_class + fn_class + eps
        )

        macro_f1 = f1_per_class.mean()

        return micro_f1.item(), macro_f1.item()

    # ------------------------------------------------------------------
    # Hyperparameter search space
    # ------------------------------------------------------------------

    search_space = {
        "num_layers": [2, 3, 4],
        "num_filters": [16, 32, 64, 128],
        "kernel_size": [3, 5, 7],
        "dropout_rate": [0.1, 0.2, 0.3, 0.4, 0.5],
    }

    all_combinations = list(itertools.product(
        search_space["num_layers"],
        search_space["num_filters"],
        search_space["kernel_size"],
        search_space["dropout_rate"],
    ))

    random.shuffle(all_combinations)

    if n_trials > len(all_combinations):
        n_trials = len(all_combinations)

    sampled_combinations = all_combinations[:n_trials]

    print(f"Total possible combinations: {len(all_combinations)}", flush=True)
    print(f"Running random search with {n_trials} trials", flush=True)

    # ------------------------------------------------------------------
    # Training setup
    # ------------------------------------------------------------------

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}", flush=True)

    criterion = nn.BCEWithLogitsLoss()

    results = []
    best_val_macro_f1 = -1.0
    best_val_loss = float("inf")
    best_model_path = ckpt_root / "best_random_search_cnn.pt"
    results_path = ckpt_root / "random_search_results.csv"

    scaler = torch.cuda.amp.GradScaler(
        enabled=(use_mixed_precision and torch.cuda.is_available())
    )

    # ------------------------------------------------------------------
    # Trial loop
    # ------------------------------------------------------------------

    for trial_idx, (num_layers, num_filters, kernel_size, dropout_rate) in enumerate(sampled_combinations, start=1):
        trial_start = time.time()

        print(
            "\n"
            f"Trial {trial_idx}/{n_trials} | "
            f"layers={num_layers}, "
            f"filters={num_filters}, "
            f"kernel={kernel_size}, "
            f"dropout={dropout_rate}",
            flush=True,
        )

        model = FlexibleCNN(
            num_layers=num_layers,
            num_filters=num_filters,
            kernel_size=kernel_size,
            num_classes=num_classes,
            batch_norm_included=False,
            image_size=image_size,
            dropout_rate=dropout_rate,
        ).to(device)

        optimizer = torch.optim.Adam(model.parameters(), lr=lr)

        best_trial_val_loss = float("inf")
        best_trial_micro_f1 = 0.0
        best_trial_macro_f1 = 0.0

        for epoch in range(1, epochs_per_trial + 1):
            epoch_start = time.time()

            # -------------------------------
            # Training
            # -------------------------------
            model.train()

            train_loss_sum = 0.0
            train_num_samples = 0

            for batch_idx, (images, targets) in enumerate(train_loader):
                images = images.to(device, non_blocking=True)
                targets = targets.to(device, non_blocking=True)

                optimizer.zero_grad(set_to_none=True)

                with torch.cuda.amp.autocast(
                    enabled=(use_mixed_precision and torch.cuda.is_available())
                ):
                    logits = model(images)
                    loss = criterion(logits, targets)

                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()

                batch_size_actual = images.size(0)
                train_loss_sum += loss.item() * batch_size_actual
                train_num_samples += batch_size_actual

                if batch_idx % 50 == 0:
                    print(
                        f"Trial {trial_idx} | Epoch {epoch}/{epochs_per_trial} | "
                        f"Batch {batch_idx}/{len(train_loader)} | "
                        f"loss={loss.item():.4f}",
                        flush=True,
                    )

            train_loss = train_loss_sum / train_num_samples

            # -------------------------------
            # Validation
            # -------------------------------
            model.eval()

            val_loss_sum = 0.0
            val_num_samples = 0

            all_val_logits = []
            all_val_targets = []

            with torch.no_grad():
                for images, targets in val_loader:
                    images = images.to(device, non_blocking=True)
                    targets = targets.to(device, non_blocking=True)

                    with torch.cuda.amp.autocast(
                        enabled=(use_mixed_precision and torch.cuda.is_available())
                    ):
                        logits = model(images)
                        loss = criterion(logits, targets)

                    batch_size_actual = images.size(0)
                    val_loss_sum += loss.item() * batch_size_actual
                    val_num_samples += batch_size_actual

                    all_val_logits.append(logits.detach().cpu())
                    all_val_targets.append(targets.detach().cpu())

            val_loss = val_loss_sum / val_num_samples

            all_val_logits = torch.cat(all_val_logits, dim=0)
            all_val_targets = torch.cat(all_val_targets, dim=0)

            val_micro_f1, val_macro_f1 = compute_multilabel_f1_from_logits(
                logits=all_val_logits,
                targets=all_val_targets,
                threshold=0.5,
            )

            if val_loss < best_trial_val_loss:
                best_trial_val_loss = val_loss
                best_trial_micro_f1 = val_micro_f1
                best_trial_macro_f1 = val_macro_f1

            print(
                f"Trial {trial_idx} | Epoch {epoch}/{epochs_per_trial} finished | "
                f"train_loss={train_loss:.4f} | "
                f"val_loss={val_loss:.4f} | "
                f"val_micro_f1={val_micro_f1:.4f} | "
                f"val_macro_f1={val_macro_f1:.4f} | "
                f"epoch_time={time.time() - epoch_start:.1f}s",
                flush=True,
            )

        trial_seconds = time.time() - trial_start

        row = {
            "trial": trial_idx,
            "num_layers": num_layers,
            "num_filters": num_filters,
            "kernel_size": kernel_size,
            "dropout_rate": dropout_rate,
            "epochs_per_trial": epochs_per_trial,
            "batch_size": batch_size,
            "lr": lr,
            "image_size": image_size,
            "best_trial_val_loss": best_trial_val_loss,
            "best_trial_val_micro_f1": best_trial_micro_f1,
            "best_trial_val_macro_f1": best_trial_macro_f1,
            "trial_seconds": trial_seconds,
        }

        results.append(row)

        results_df = pd.DataFrame(results)
        results_df.to_csv(results_path, index=False)

        # Save the best global model by macro F1.
        # If macro F1 is tied, prefer lower validation loss.
        is_better = (
            best_trial_macro_f1 > best_val_macro_f1
            or (
                best_trial_macro_f1 == best_val_macro_f1
                and best_trial_val_loss < best_val_loss
            )
        )

        if is_better:
            best_val_macro_f1 = best_trial_macro_f1
            best_val_loss = best_trial_val_loss

            torch.save(
                {
                    "model_state_dict": model.state_dict(),
                    "classes": classes,
                    "num_classes": num_classes,
                    "image_size": image_size,
                    "best_val_loss": best_val_loss,
                    "best_val_macro_f1": best_val_macro_f1,
                    "hyperparameters": {
                        "num_layers": num_layers,
                        "num_filters": num_filters,
                        "kernel_size": kernel_size,
                        "dropout_rate": dropout_rate,
                        "lr": lr,
                        "batch_size": batch_size,
                        "epochs_per_trial": epochs_per_trial,
                    },
                },
                best_model_path,
            )

            print(f"Saved new best model to {best_model_path}", flush=True)

        try:
            ckpt_vol.commit()
        except Exception:
            pass

    results_df = pd.DataFrame(results).sort_values(
        by=["best_trial_val_macro_f1", "best_trial_val_loss"],
        ascending=[False, True],
    )

    results_df.to_csv(results_path, index=False)

    try:
        ckpt_vol.commit()
    except Exception:
        pass

    best_row = results_df.iloc[0].to_dict()

    return {
        "results_path": str(results_path),
        "best_model_path": str(best_model_path),
        "best_trial": best_row,
        "num_trials": int(len(results_df)),
    }


## Optional: inspect saved search results

This function reads `/ckpt/random_search_results.csv` from the Modal checkpoint volume and returns it as a list of dictionaries.


In [23]:
@app.function(
    image=image,
    volumes={"/ckpt": ckpt_vol},
)
def read_random_search_results():
    from pathlib import Path
    import pandas as pd

    results_path = Path("/ckpt/random_search_results.csv")

    if not results_path.exists():
        raise FileNotFoundError(f"Could not find {results_path}")

    df = pd.read_csv(results_path)
    return df.to_dict(orient="records")


In [24]:
if __name__ == "__main__":
    with app.run():
        metadata = prepare_data.remote()
        print("Dataset metadata:", metadata)

        search_results = random_search_cnn.remote(
            n_trials=40,
            epochs_per_trial=1,
            batch_size=64,
            lr=1e-4,
            image_size=224,
            num_workers=8,
            prefetch_factor=2,
            seed=42,
            use_mixed_precision=True,
        )

        print("Random search finished.")
        print(search_results)


Dataset metadata: {'num_train': 15837, 'num_val': 2795, 'classes': ['complex', 'frog_eye_leaf_spot', 'healthy', 'powdery_mildew', 'rust', 'scab'], 'num_classes': 6}
Random search finished.
{'results_path': '/ckpt/random_search_results.csv', 'best_model_path': '/ckpt/best_random_search_cnn.pt', 'best_trial': {'trial': 23.0, 'num_layers': 2.0, 'num_filters': 32.0, 'kernel_size': 7.0, 'dropout_rate': 0.1, 'epochs_per_trial': 1.0, 'batch_size': 64.0, 'lr': 0.0001, 'image_size': 224.0, 'best_trial_val_loss': 0.37820843043395574, 'best_trial_val_micro_f1': 0.11800302565097809, 'best_trial_val_macro_f1': 0.13863039016723633, 'trial_seconds': 334.4258131980896}, 'num_trials': 24}


In [25]:
# Optional: read the saved CSV back from the Modal checkpoint volume.
if __name__ == "__main__":
    with app.run():
        rows = read_random_search_results.remote()
        print(f"Loaded {len(rows)} result rows")
        for row in rows[:5]:
            print(row)


Loaded 24 result rows
{'trial': 23, 'num_layers': 2, 'num_filters': 32, 'kernel_size': 7, 'dropout_rate': 0.1, 'epochs_per_trial': 1, 'batch_size': 64, 'lr': 0.0001, 'image_size': 224, 'best_trial_val_loss': 0.3782084304339557, 'best_trial_val_micro_f1': 0.118003025650978, 'best_trial_val_macro_f1': 0.1386303901672363, 'trial_seconds': 334.4258131980896}
{'trial': 35, 'num_layers': 2, 'num_filters': 64, 'kernel_size': 7, 'dropout_rate': 0.5, 'epochs_per_trial': 1, 'batch_size': 64, 'lr': 0.0001, 'image_size': 224, 'best_trial_val_loss': 0.3796885605041797, 'best_trial_val_micro_f1': 0.1433649361133575, 'best_trial_val_macro_f1': 0.1342893093824386, 'trial_seconds': 313.8169529438019}
{'trial': 34, 'num_layers': 3, 'num_filters': 128, 'kernel_size': 3, 'dropout_rate': 0.1, 'epochs_per_trial': 1, 'batch_size': 64, 'lr': 0.0001, 'image_size': 224, 'best_trial_val_loss': 0.3774765779379229, 'best_trial_val_micro_f1': 0.0881166607141494, 'best_trial_val_macro_f1': 0.1328265219926834, 'trial